<a href="https://colab.research.google.com/github/bsudharsh1979/NATcolab/blob/main/notebooks/1_ALM_Application_Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**NATcolab** · NVIDIA NeMo Agent Toolkit · *Asset Lifecycle Management with Agentic AI* — Google Colab edition (hosted NVIDIA NIMs)

# Asset Lifecycle Management - Basic Interaction & Frontend

> **🟢 Colab edition.** This is the Google Colab adaptation of the NVIDIA DLI lab notebook. Instead of the lab's Docker containers it uses:
> - **Hosted NVIDIA NIMs** on build.nvidia.com — `nvidia/nemotron-3-ultra-550b-a55b` (planning, tool calling, text-to-SQL), `nvidia/nemotron-3.5-lightning-30b-a3b` (code generation), `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning` (multimodal judge), `nvidia/nemotron-3.5-embed-1b` (embeddings)
> - **MOMENT-1-small**, an open-source time-series foundation model that runs locally, in place of the self-hosted NV-Tesseract NIM
> - a local Python **code sandbox** and a **Phoenix** server started from the notebook
>
> **Before you run:** add your NVIDIA API key (get one at [build.nvidia.com](https://build.nvidia.com)) as a Colab secret named `NVIDIA_API_KEY` — 🔑 icon in the left sidebar → *Add new secret* → enable *Notebook access*. A CPU runtime is enough.
> Colab gives every notebook its own VM, so run the **setup cell below at the top of every notebook** (≈5–8 min on a fresh runtime, seconds afterwards). Paths that were `/dli/task/...` in the lab are `/content/NATcolab/...` here.

> **Prerequisites:** none in Colab — the setup cell builds the database (Notebook 0) automatically.

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Start the NAT backend service** and verify it's running correctly
2. **Create an API client** to communicate with the agent
3. **Send queries** to the agent and receive responses
4. **Understand Chain-of-Thought reasoning** in agent responses
5. **Monitor the agent interactively** (in Colab: stream intermediate steps into the notebook instead of the NAT Toolkit UI)
6. **Enable intermediate step visualization** to see agent reasoning in real-time
7. **Run progressively complex queries** and observe agent behavior

In [ ]:
# ⚙️ NATcolab setup — run first (≈5–8 min on a fresh runtime; a few seconds if already done in this session)
# Clones github.com/bsudharsh1979/NATcolab, creates the NAT virtual env (nvidia-nat 1.2.1 + ALM agent),
# builds the NASA turbofan database, trains the RUL model and checks the hosted NVIDIA NIMs.
import os, sys, subprocess
ALM_HOME = os.environ.get("ALM_HOME", "/content/NATcolab")
if not os.path.exists(f"{ALM_HOME}/colab_bootstrap.py"):
    subprocess.run(["git", "clone", "-q", "https://github.com/bsudharsh1979/NATcolab.git", ALM_HOME], check=True)
sys.path.insert(0, ALM_HOME)
os.chdir(ALM_HOME)

import colab_bootstrap
colab_bootstrap.bootstrap()

In [ ]:
from nat_client import NATClient, link
client = NATClient()

---
# Understanding Agent Interaction

## How Agentic AI Systems Work

Before we start interacting with our agent, let's understand how the conversation flow works:

```
User Query
    │
    ▼
┌──────────────────────────────────────────┐
│ Reasoning Agent (Planner)                │
│ - Analyzes query intent                  │
│ - Breaks down into steps                 │
│ - Creates execution plan                 │
└──────────────┬───────────────────────────┘
               │ Plan
               ▼
┌──────────────────────────────────────────┐
│ Data Analysis Agent (Executor)           │
│ Step 1: Retrieve data → SQL Tool         │
│ Step 2: Analyze patterns → ML Tool       │
│ Step 3: Visualize → Plotting Tool        │
└──────────────┬───────────────────────────┘
               │ Results
               ▼
         Final Response
         (with visualizations)
```

## Chain-of-Thought Reasoning

The agent uses **Chain-of-Thought (CoT)** reasoning:
- Breaks complex problems into steps
- Explains its reasoning process
- Shows tool usage and intermediate results
- Arrives at a final answer with evidence

Example:
```
Query: "Which engine in FD001 has the shortest remaining life?"

Agent Reasoning:
1. Thought: I need to find engines with minimum RUL in FD001
2. Action: Use SQL retrieval tool
3. Observation: Query returned unit 81 with RUL=5
4. Final Answer: Engine unit 81 has the shortest remaining life (5 cycles)
```

---
# NVIDIA NeMo Agent Toolkit (NAT)

## What is NAT?

**NeMo Agent Toolkit (NAT)** is NVIDIA's framework for building production-ready agentic AI applications. NAT provides:

- **Modular agent architectures**: ReAct agents, reasoning agents, code generation agents
- **Tool integration**: Easy-to-define custom tools and functions
- **LLM flexibility**: Support for NVIDIA NIMs, OpenAI, and other LLM providers
- **Observability**: Built-in tracing and monitoring
- **Evaluation**: Custom evaluators for agentic workflows
- **Production-ready**: Configuration-driven, scalable architecture

## NAT System Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                      User Interface                         │
│              (Jupyter / Web UI / API Client)                │
└────────────────────────┬────────────────────────────────────┘
                         │ HTTP/REST API
                         ▼
┌─────────────────────────────────────────────────────────────┐
│                   NAT Backend Service                       │
│  ┌──────────────────────────────────────────────────────┐   │
│  │           Reasoning Agent (Planner)                  │   │
│  │  - Analyzes user query                               │   │
│  │  - Creates execution plan                            │   │
│  │  - Uses: Nemotron/Llama models                       │   │
│  └──────────────────────┬───────────────────────────────┘   │
│                         │                                   │
│                         ▼                                   │
│  ┌──────────────────────────────────────────────────────┐   │
│  │      Data Analysis Agent (Executor - ReAct)          │   │
│  │  - Executes plan step-by-step                        │   │
│  │  - Calls tools iteratively                           │   │
│  │  - Uses: Qwen/specialized models                     │   │
│  └──────────────────────┬───────────────────────────────┘   │
│                         │                                   │
│                         ▼                                   │
│  ┌──────────────────────────────────────────────────────┐   │
│  │                Tool Functions                        │   │
│  │  ┌────────────┐  ┌────────────┐  ┌────────────┐      │   │
│  │  │ SQL Query  │  │ Predict RUL│  │Plot Charts │      │   │
│  │  │  (Vanna)   │  │  (XGBoost) │  │(Matplotlib)│      │   │
│  │  └────────────┘  └────────────┘  └────────────┘      │   │
│  │  ┌────────────┐  ┌────────────┐  ┌────────────┐      │   │
│  │  │  Anomaly   │  │Code Gen &  │  │Recommend   │      │   │
│  │  │ (MOMENT)   │  │ Execution  │  │  Actions   │      │   │
│  │  └────────────┘  └────────────┘  └────────────┘      │   │
│  └──────────────────────────────────────────────────────┘   │
└────────────────────────┬────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────┐
│                  External Services                          │
│  ┌────────────┐  ┌────────────┐  ┌────────────┐             │
│  │ SQLite DB  │  │Phoenix/W&B │  │ Code       │             │
│  │ (NASA Data)│  │(Monitoring)│  │ Sandbox    │             │
│  └────────────┘  └────────────┘  └────────────┘             │
└─────────────────────────────────────────────────────────────┘
```

## Key Components

1. **Reasoning Agent**: High-level planner that breaks down complex queries into steps
2. **Data Analysis Agent**: ReAct-style agent that executes the plan using tools
3. **Tools**: Modular functions (SQL queries, ML predictions, visualizations, etc.)
4. **LLMs**: Multiple specialized models for different tasks (reasoning, coding, analysis)
5. **Configuration**: YAML-based config defining agents, tools, and their connections

### Models used in this Colab edition

| Role in `config-reasoning.yml` | Hosted NVIDIA NIM |
|---|---|
| `reasoning_llm` — planner (reasoning agent) | `nvidia/nemotron-3-ultra-550b-a55b` |
| `analyst_llm` — ReAct executor / tool calling | `nvidia/nemotron-3-ultra-550b-a55b` |
| `sql_llm` — Vanna text-to-SQL | `nvidia/nemotron-3-ultra-550b-a55b` |
| `coding_llm` — code generation (Optional notebook) | `nvidia/nemotron-3.5-lightning-30b-a3b` |
| `multimodal_judging_llm` — evaluation judge | `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning` |
| `vanna_embedder` — embeddings | `nvidia/nemotron-3.5-embed-1b` |
| anomaly detection | MOMENT-1-small (local, open source) |

---
# Section 1: Install the NAT Agent Package

Before starting the backend, we need to install the **Asset Lifecycle Management agent** — the Python package that defines all the tools, agents, and configurations for this course.

## Understanding `pyproject.toml`

The agent package lives in `asset_lifecycle_management_agent/` and is structured as a standard Python project with a `pyproject.toml` file. This file tells NAT (and Python) three important things:

| Section | What it controls |
|---|---|
| **`[project]`** | Package name, version, and description |
| **`[project.dependencies]`** | All libraries the agent needs (NAT framework, LangChain, XGBoost, Vanna, MOMENT, etc.) |
| **`[project.entry-points]`** | How NAT discovers tools — each tool class is registered here so `nat serve` can find and load it from the config |

When you run `uv pip install -e .`, Python installs the package in **editable mode** (`-e`) — meaning any changes you make to the tool source files take effect immediately without reinstalling.

## 1.1 Install Dependencies

In [ ]:
# Re-install the agent package in editable mode into the NAT venv (fast — it is already installed by the setup cell)
!cd {ALM_HOME}/asset_lifecycle_management_agent && uv pip install --python {client.python_bin} -q -e .

## 1.2 Verify Installation

In [ ]:
import subprocess, sys

deps = ["nvidia-nat", "nvidia-nat-langchain", "nvidia-nat-phoenix", "vanna", "chromadb", "xgboost", "momentfm", "torch"]
installed = subprocess.run(["uv", "pip", "list", "--python", client.python_bin], capture_output=True, text=True).stdout.lower()

missing = sum(1 for d in deps if d.lower() not in installed)
for d in deps:
    print(f"{'✅' if d.lower() in installed else '❌'} {d}")

print(f"\n📊 {len(deps)-missing}/{len(deps)} installed | NAT venv: {client.python_bin}")

---
# Section 2: Start the NAT Backend Service

The NAT backend service runs as a REST API server that handles all agent requests.

## 2.1 Start Backend Service

NAT provides a CLI command to launch the backend workflow server using the `nat serve` command. On a machine with a terminal (Colab Pro's *Terminal*, or your own Linux box) you can start it manually:

```bash
cd /content/NATcolab
source /content/alm-venv/bin/activate
nat serve --config_file=configs/config-reasoning.yml
```

To make things easier within your notebook, the Python client controls the server directly from a notebook cell. `client.start()` launches `nat serve` in the background **and waits until it is healthy** (the first start trains the Vanna SQL retriever with NIM embeddings and loads MOMENT — allow 1–3 minutes):

In [ ]:
client.start()

In [ ]:
client.logs(20)

This client will start the backend, monitor health, and help you troubleshoot without leaving your notebook.

**Wait for the service to start** before proceeding to the next section.

If all goes well, you will see:

```
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:8000 (Press CTRL+C to quit)
```

### What's Happening?

The `nat serve` command:
1. Loads the configuration from `config-reasoning.yml`
2. Initializes all LLMs (reasoning, analyst, SQL, coding models)
3. Registers all tools (SQL retrieval, RUL prediction, plotting, anomaly detection)
4. Sets up the reasoning agent and data analysis agent
5. Starts a FastAPI server on port 8000

### Troubleshooting

If you see errors:
- **"Port already in use"**: Another service is running on port 8000. Kill it with `fuser -k 8000/tcp`, or from a notebook: `client.stop()`
- **"Model not found"**: Check that NVIDIA_API_KEY is set correctly
- **"Database not found"**: Ensure you completed Notebook 0's database setup

---
# Section 3: Basic Interaction with the Agent

Now let's establish communication with our agent through Python.

## 3.1 Health Check

Before sending queries, let's verify the backend service is running:

In [ ]:
client.check_health()

## 3.2 Simple Queries

Let's start with progressively complex queries to understand how the agent works.

### Query 1: Domain Knowledge (No Tools Needed)

In [ ]:
response = client.query("What can you do?")

**What Happened?**
- The agent answered using its built-in knowledge
- No tools were called
- This is a pure LLM response

### Query 2: Data Retrieval (SQL Tool)

In [ ]:
response = client.query("What is the ground truth RUL of unit_number 20 in dataset FD001?")

**What Happened?**
- The reasoning agent created a plan: "Use SQL tool to query RUL"
- The data analysis agent executed the SQL query
- The result was returned to the user

**Note:** You can't see the intermediate steps yet. We'll enable that in Section 4!

### Query 3: Complex Multi-Step Query

In [ ]:
response = client.query("Show me a line chart with sensor_measurement_4 values for unit 69 in FD001 training data over time")

**What Happened?**
- SQL tool retrieved sensor data
- Plotting tool created a visualization
- The agent returned both text and file path to the plot

Let's display the generated chart directly in the notebook:

In [ ]:
# Display the generated chart (the plotting tool saves line_chart_<x>_vs_<y>.png/.html in agent_workspace/)
client.show("line_chart_time_in_cycles_vs_sensor_measurement_4.png")

---
# Section 4: Watching the Agent Think — Intermediate Steps

In the DLI lab, Section 4 starts the **NAT Toolkit UI** (a React/Next.js app on port 3000) to visualize intermediate steps. That UI needs a long-lived Node.js dev server and direct browser access to the backend, which Colab's sandbox does not provide, so this edition streams the same **intermediate steps** straight into the notebook instead:

- The reasoning agent's plan
- Each tool invocation and its input
- Intermediate results
- The final answer

`client.query(..., show_steps=True)` calls the backend's streaming endpoint (`/chat/stream`) and renders every intermediate step as a collapsible block. (Running the course on your own machine? The [NeMo-Agent-Toolkit-UI](https://github.com/NVIDIA/NeMo-Agent-Toolkit-UI) works unchanged against `nat serve` on port 8000.)

In [ ]:
response = client.query("What is the ground truth RUL of unit_number 20 in dataset FD001?", show_steps=True)

## 4.3 Configure Intermediate Step Visualization

Once you open the UI:

1. **Click the Settings icon** (bottom right corner)
2. **Enable these options:**
   - ✅ Enable Intermediate Steps
   - ✅ Expand Intermediate Steps by Default
3. **Close the settings panel**

These settings allow you to see:
- The reasoning agent's plan
- Each tool invocation
- Intermediate results
- The final answer

### Understanding Intermediate Steps

With intermediate steps enabled, you'll see:

```
🧠 Reasoning Agent
  Plan:
  1. Retrieve sensor data using SQL
  2. Plot the data
  3. Return results

🔧 Data Analysis Agent
  Step 1: SQL Retrieval
    Input: {table: train_FD001, unit: 5, column: sensor_measurement_1}
    Output: Retrieved 192 rows
  
  Step 2: Plot Line Chart
    Input: {x: time_in_cycles, y: sensor_measurement_1}
    Output: Plot saved to output_data/plot_12345.png

✅ Final Answer
  Here is the sensor reading plot for unit 5...
  [Plot visualization shown inline]
```

## 4.4 Test Queries with Intermediate Steps

Try the queries from previous sections with `show_steps=True` and observe the plan and tool calls.

---
# Exercise: Write Custom Queries

**Objective:** Practice writing queries and understanding agent responses.

**Tasks:**
1. Write 3 custom queries of increasing complexity
2. Run them in the UI with intermediate steps enabled
3. Document the tools used and reasoning steps

**Hints:**
- Start simple (data retrieval only)
- Add visualization (retrieval + plotting)
- Try complex (retrieval + analysis + recommendation)


In [ ]:
# Exercise: write 3 custom queries of increasing complexity (edit freely!)
my_queries = [
    "How many engines are in the FD003 training dataset?",                                    # retrieval only
    "Plot sensor_measurement_11 vs time_in_cycles for unit 5 in FD001 training data",         # retrieval + plot
    "Retrieve sensor data for unit 24 in FD001 test data and predict its remaining useful life",  # retrieval + prediction
]
for q in my_queries:
    client.query(q, show_steps=True)

---
# Key Takeaways

In this notebook, you learned:

1. ✅ How to start the NAT backend service
2. ✅ How to create a Python API client
3. ✅ How to send queries programmatically
4. ✅ How to stream the agent's intermediate steps into the notebook
5. ✅ How to enable and interpret intermediate reasoning steps
6. ✅ How the agent uses different tools for different query types
7. ✅ How query complexity affects agent reasoning

## What's Next?

In **Notebook 2**, you will:
- Modify agent configurations (temperature, prompts)
- Create custom tools (color preferences, action recommendations)
- Understand the NAT configuration file structure
- Learn how to extend agent capabilities

**[Continue to Notebook 2 - Configuration & Tools →](2_ALM_Config_Understanding.ipynb)**